# 16. Agregación de señales local-plato — Sevilla

Este notebook transforma las menciones de platos con sentimiento en señales agregadas por `place_id + dish_id_v1`.

Entrada:

```text
data/artifacts/ai/sevilla/sentiment/sevilla_dish_mentions_with_sentiment_v1.jsonl
```

Salidas:

```text
data/artifacts/ai/sevilla/aggregation/sevilla_place_dish_signals_v1.csv
data/artifacts/ai/sevilla/aggregation/sevilla_place_dish_signals_v1.jsonl
data/artifacts/ai/sevilla/aggregation/sevilla_place_dish_ranking_candidates_v1.csv
data/artifacts/ai/sevilla/aggregation/sevilla_global_dish_signals_v1.csv
data/artifacts/ai/sevilla/aggregation/sevilla_place_dish_signal_summary_v1.json
```

In [5]:
# ============================================================
# 01. Imports y rutas
# ============================================================

from __future__ import annotations

import ast
import json
import math
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 180)
pd.set_option("display.width", 220)

CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR

INPUT_PATH = PROJECT_ROOT / "data" / "artifacts" / "ai" / "sevilla" / "sentiment" / "sevilla_dish_mentions_with_sentiment_v1.jsonl"
OUTPUT_DIR = PROJECT_ROOT / "data" / "artifacts" / "ai" / "sevilla" / "aggregation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("INPUT_PATH:", INPUT_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("Existe input:", INPUT_PATH.exists())

PROJECT_ROOT: c:\Users\USUARIO\Documents\Proyectos_Master_IA_Big_Data\hidden-gems-pipeline
INPUT_PATH: c:\Users\USUARIO\Documents\Proyectos_Master_IA_Big_Data\hidden-gems-pipeline\data\artifacts\ai\sevilla\sentiment\sevilla_dish_mentions_with_sentiment_v1.jsonl
OUTPUT_DIR: c:\Users\USUARIO\Documents\Proyectos_Master_IA_Big_Data\hidden-gems-pipeline\data\artifacts\ai\sevilla\aggregation
Existe input: True


In [6]:
# ============================================================
# 02. Utilidades
# ============================================================

def read_jsonl(path: Path) -> pd.DataFrame:
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for i, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(f"JSON inválido en línea {i}: {exc}") from exc
    return pd.DataFrame(rows)


def save_jsonl(df: pd.DataFrame, path: Path) -> None:
    records = df.replace({np.nan: None}).to_dict(orient="records")
    with path.open("w", encoding="utf-8") as f:
        for row in records:
            f.write(json.dumps(row, ensure_ascii=False, default=str) + "\n")


def save_json(data: dict[str, Any], path: Path) -> None:
    with path.open("w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2, default=str)


def parse_maybe_list(value: Any) -> list[str]:
    if isinstance(value, list):
        return [str(x) for x in value]
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return []
    if isinstance(value, str):
        txt = value.strip()
        if not txt:
            return []
        try:
            parsed = ast.literal_eval(txt)
            if isinstance(parsed, list):
                return [str(x) for x in parsed]
            return [str(parsed)]
        except Exception:
            if "," in txt:
                return [x.strip() for x in txt.split(",") if x.strip()]
            return [txt]
    return [str(value)]


def clip01(value: float) -> float:
    return float(max(0.0, min(1.0, value)))


def sigmoid_like(value: float, scale: float = 2.6) -> float:
    return float((math.tanh(value / scale) + 1.0) / 2.0)

In [7]:
# ============================================================
# 03. Carga y validación
# ============================================================

if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f"No se encontró {INPUT_PATH}. Ejecuta antes el notebook 15_sevilla_mention_sentiment.ipynb."
    )

mentions_df = read_jsonl(INPUT_PATH)

print("Shape:", mentions_df.shape)
display(mentions_df.head(3))

REQUIRED_COLUMNS = [
    "mention_id", "dish_id_v1", "review_id", "place_id", "place_name",
    "district_id", "district_name", "neighborhood_id", "neighborhood_name",
    "rating_value", "display_dish_name_es_v1", "normalized_dish_name_es_v1",
    "dish_family_es_v1", "dish_group_es_v1", "mention_sentiment_label_v1",
    "mention_sentiment_score_v1", "mention_sentiment_confidence_v1",
    "mention_sentiment_weight_v1", "sentiment_reason_v1",
]

missing = [c for c in REQUIRED_COLUMNS if c not in mentions_df.columns]
if missing:
    raise KeyError(f"Faltan columnas obligatorias: {missing}")

basic_checks = {
    "has_rows": bool(len(mentions_df) > 0),
    "mention_ids_are_unique": bool(mentions_df["mention_id"].nunique(dropna=True) == len(mentions_df)),
    "all_have_place": bool(mentions_df["place_id"].notna().all()),
    "all_have_dish": bool(mentions_df["dish_id_v1"].notna().all()),
    "all_have_review": bool(mentions_df["review_id"].notna().all()),
    "all_have_sentiment_label": bool(mentions_df["mention_sentiment_label_v1"].notna().all()),
}

print(json.dumps(basic_checks, indent=2, ensure_ascii=False))

Shape: (2979, 59)


,mention_id,dish_id_v1,review_id,place_id,place_source_ref_id,source_review_id,source_place_record_id,place_name,district_id,district_name,neighborhood_id,neighborhood_name,rating_value,review_sentiment_from_rating,review_language,dish_mention_text,dish_mention_normalized,canonical_dish_name_es_v1,normalized_dish_name_es_v1,display_dish_name_es_v1,dish_group_es_v1,dish_family_es_v1,dish_specificity_v1,candidate_type,lexicon_group,detection_method,pattern_name,confidence,mention_quality_tier_v1,ranking_eligible_v1,eligibility_status_v1,eligibility_reason_v1,needs_manual_review_v1,normalization_method_v1,start_char,end_char,context_sentence,window_context,has_order_trigger,has_positive_context,has_negative_context,has_contrast_context,text,mention_sentiment_label_v1,mention_sentiment_score_v1,mention_sentiment_local_score_v1,mention_sentiment_local_strength_v1,mention_sentiment_confidence_v1,mention_sentiment_weight_v1,sentiment_reason_v1,sentiment_flags_v1,positive_terms_v1,negative_terms_v1,neutral_terms_v1,rating_prior_label_v1,rating_prior_score_v1,context_used_v1,context_source_v1,debug_sentiment_matches_v1
0,1fde28a5f80c28e3948ba1bf865c0b74017762f5,dish_es_v1_d5782aef2e608777,0d0946cb-a72c-4197-9d43-b5f3dbfcc620,8cb19db3-e9a8-4a42-be63-73e86e22a3d9,2acb2538-7f10-4760-a646-85c41436f9e8,a299c238d5349ed3eb5fc64b5db7e6def4db213c4ed81730074b12931536495c,ChIJ50jgAn5tEg0RBySkaJ7I8eE,Tropitali,5bd4fbf1-457b-4dcb-91f1-089ac449a5db,Casco Antiguo,e554ef0e-d9f1-4d89-8777-f92cee028f7f,SAN JULIAN,5,positive,es,Açai,acai,acai,acai,acai,internacional,internacional,clear_single,dish_candidate,dish_core,exact_alias,NaN,0.83,medium,True,eligible,dish_like_candidate,False,canonical_passthrough,19,23,"Sin dudas el mejor Açai que he probado aqui en Sevilla, la coxinha muy rica y el trato del personal increíble, son muy muy muy amables.","Sin dudas el mejor Açai que he probado aqui en Sevilla, la coxinha muy rica y el trato del personal increíble, son muy muy muy amables. Gracias por traer un pedaci",False,True,False,False,"Sin dudas el mejor Açai que he probado aqui en Sevilla, la coxinha muy rica y el trato del personal increíble, son muy muy muy amables. Gracias por traer un pedacito de Brasil ...",positive,3.6520,3.6520,3.6520,0.88,0.8800,local_context_primary,"[intensifier_nearby, sentiment_may_refer_to_service]","[increible:positive_lexicon, rica:positive_lexicon]",[],[],positive,1.15,"Sin dudas el mejor Açai que he probado aqui en Sevilla, la coxinha muy rica y el trato del personal increíble, son muy muy muy amables.",context_sentence,"[increible@99:2.00, rica@70:1.65]"
1,b70f05c16925cdb70c7f0ea3f4ce0d328d3ccc24,dish_es_v1_987dcc3d8e157f2d,0d0946cb-a72c-4197-9d43-b5f3dbfcc620,8cb19db3-e9a8-4a42-be63-73e86e22a3d9,2acb2538-7f10-4760-a646-85c41436f9e8,a299c238d5349ed3eb5fc64b5db7e6def4db213c4ed81730074b12931536495c,ChIJ50jgAn5tEg0RBySkaJ7I8eE,Tropitali,5bd4fbf1-457b-4dcb-91f1-089ac449a5db,Casco Antiguo,e554ef0e-d9f1-4d89-8777-f92cee028f7f,SAN JULIAN,5,positive,es,coxinha,coxinha,coxinha,coxinha,coxinha,internacional,internacional,clear_single,dish_candidate,dish_core,exact_alias,NaN,0.83,medium,True,eligible,dish_like_candidate,False,canonical_passthrough,59,66,"Sin dudas el mejor Açai que he probado aqui en Sevilla, la coxinha muy rica y el trato del personal increíble, son muy muy muy amables.","Sin dudas el mejor Açai que he probado aqui en Sevilla, la coxinha muy rica y el trato del personal increíble, son muy muy muy amables. Gracias por traer un pedacito de Brasil ...",False,True,False,False,"Sin dudas el mejor Açai que he probado aqui en Sevilla, la coxinha muy rica y el trato del personal increíble, son muy muy muy amables. Gracias por traer un pedacito de Brasil ...",positive,4.3094,4.3094,4.3094,0.88,0.8800,local_context_primary,"[intensifier_nearby, sentiment_may_refer_to_service]","[increible:positive_lexicon, rica:positive_lexicon]",[],[],positive,1.15,"Sin dudas el mejor Açai que he probado aqui en Sevilla, la coxinha muy rica

{
  "has_rows": true,
  "mention_ids_are_unique": true,
  "all_have_place": true,
  "all_have_dish": true,
  "all_have_review": true,
  "all_have_sentiment_label": true
}


In [8]:
# ============================================================
# 04. Preparación de variables para agregación
# ============================================================

df = mentions_df.copy()

for col in [
    "rating_value", "confidence", "mention_sentiment_score_v1",
    "mention_sentiment_local_score_v1", "mention_sentiment_local_strength_v1",
    "mention_sentiment_confidence_v1", "mention_sentiment_weight_v1",
]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

df["mention_sentiment_label_v1"] = (
    df["mention_sentiment_label_v1"].fillna("neutral").astype(str).str.lower().str.strip()
)

valid_labels = {"positive", "neutral", "negative"}
df.loc[~df["mention_sentiment_label_v1"].isin(valid_labels), "mention_sentiment_label_v1"] = "neutral"

df["sentiment_value_v1"] = df["mention_sentiment_label_v1"].map(
    {"positive": 1.0, "neutral": 0.0, "negative": -1.0}
).fillna(0.0)

if "sentiment_flags_v1" in df.columns:
    df["sentiment_flags_list_v1"] = df["sentiment_flags_v1"].apply(parse_maybe_list)
else:
    df["sentiment_flags_list_v1"] = [[] for _ in range(len(df))]

df["has_low_confidence_flag_v1"] = df["sentiment_flags_list_v1"].apply(
    lambda xs: "low_sentiment_confidence" in xs or "weak_local_sentiment_signal" in xs
)
df["has_service_flag_v1"] = df["sentiment_flags_list_v1"].apply(lambda xs: "sentiment_may_refer_to_service" in xs)
df["has_price_flag_v1"] = df["sentiment_flags_list_v1"].apply(lambda xs: "price_context_detected" in xs)
df["has_contrast_flag_v1"] = df["sentiment_flags_list_v1"].apply(lambda xs: "contrast_marker_nearby" in xs)
df["has_negation_flag_v1"] = df["sentiment_flags_list_v1"].apply(lambda xs: "negation_detected" in xs)

df["is_local_context_v1"] = df["sentiment_reason_v1"].eq("local_context_primary")
df["is_review_prior_fallback_v1"] = df["sentiment_reason_v1"].eq("review_prior_fallback")

base_weight = df["mention_sentiment_weight_v1"].fillna(df["mention_sentiment_confidence_v1"]).fillna(0.5)
base_weight = base_weight.clip(lower=0.05, upper=1.0)

reason_multiplier = np.where(df["is_local_context_v1"], 1.00, 0.62)
service_multiplier = np.where(df["has_service_flag_v1"], 0.72, 1.00)
price_multiplier = np.where(df["has_price_flag_v1"], 0.82, 1.00)
low_conf_multiplier = np.where(df["has_low_confidence_flag_v1"], 0.78, 1.00)

df["aggregation_weight_v1"] = (
    base_weight * reason_multiplier * service_multiplier * price_multiplier * low_conf_multiplier
).clip(0.03, 1.0)

print("Dataset preparado:", df.shape)
display(df[[
    "mention_id", "place_name", "display_dish_name_es_v1", "mention_sentiment_label_v1",
    "mention_sentiment_confidence_v1", "sentiment_reason_v1", "aggregation_weight_v1",
]].head(10))

Dataset preparado: (2979, 69)


,mention_id,place_name,display_dish_name_es_v1,mention_sentiment_label_v1,mention_sentiment_confidence_v1,sentiment_reason_v1,aggregation_weight_v1
0,1fde28a5f80c28e3948ba1bf865c0b74017762f5,Tropitali,acai,positive,0.88,local_context_primary,0.63360
1,b70f05c16925cdb70c7f0ea3f4ce0d328d3ccc24,Tropitali,coxinha,positive,0.88,local_context_primary,0.63360
2,159fb380f51c372ca4a28498d1737f9b3564b0b3,Cariña VZ,arepa,positive,0.36,review_prior_fallback,0.10794
3,5fb5fba4a417980ceed5eb908918a0544b4d9550,Antojo Criollo,empanada,positive,0.36,review_prior_fallback,0.10794
4,f18cf1d71d336ce90af3ecaee466f476e42a7159,Antojo Criollo,tarta de queso,positive,0.36,review_prior_fallback,0.10794
5,7163b4138b0ccdeceb14354e15749ecb6da5ca34,Cariña VZ,arepa,positive,0.83,local_context_primary,0.59760
6,f96c4c285c116d710bc655db098ec929d9e84e6a,Healthy Nation,poke,positive,0.88,local_context_primary,0.63360
7,23bdc5adc41281a944a563a8a047ddb25e3880b7,Healthy Nation,burrito,positive,0.88,local_context_primary,0.63360
8,e20cb0dbc02ff2f90456ddc2c5e78e2705fcdbf1,Abacería LP La Pareja,morcilla,positive,0.90,local_context_primary,0.90000
9,6c3dcefa0e85a875f871740bf111bea6cf8e164f,Abacería LP La Pareja,bacalao,positive,0.90,local_context_primary,0.90000


In [9]:
# ============================================================
# 05. Agregación por local-plato
# ============================================================

GROUP_COLUMNS = [
    "place_id", "place_name", "district_id", "district_name", "neighborhood_id", "neighborhood_name",
    "dish_id_v1", "display_dish_name_es_v1", "normalized_dish_name_es_v1",
    "dish_family_es_v1", "dish_group_es_v1",
]


def aggregate_place_dish(group: pd.DataFrame) -> pd.Series:
    total = len(group)
    pos = int((group["mention_sentiment_label_v1"] == "positive").sum())
    neu = int((group["mention_sentiment_label_v1"] == "neutral").sum())
    neg = int((group["mention_sentiment_label_v1"] == "negative").sum())

    weights = group["aggregation_weight_v1"].fillna(0.05).clip(lower=0.03)
    weighted_sentiment_value = float(np.average(group["sentiment_value_v1"].fillna(0.0), weights=weights)) if total else 0.0
    weighted_sentiment_score = float(np.average(group["mention_sentiment_score_v1"].fillna(0.0), weights=weights)) if total else 0.0

    weighted_positive_count = float(((group["mention_sentiment_label_v1"] == "positive").astype(float) * weights).sum())
    weighted_neutral_count = float(((group["mention_sentiment_label_v1"] == "neutral").astype(float) * weights).sum())
    weighted_negative_count = float(((group["mention_sentiment_label_v1"] == "negative").astype(float) * weights).sum())
    total_weight = float(weights.sum())

    alpha = 1.0
    smoothed_positive_ratio = (weighted_positive_count + alpha) / (total_weight + 3 * alpha)
    smoothed_negative_ratio = (weighted_negative_count + alpha) / (total_weight + 3 * alpha)

    flags = []
    for xs in group["sentiment_flags_list_v1"]:
        flags.extend(xs)
    top_flags = [k for k, _ in Counter(flags).most_common(8)]

    example_positive = ""
    example_negative = ""
    example_neutral = ""
    if "context_used_v1" in group.columns:
        for label in ["positive", "negative", "neutral"]:
            rows = group[group["mention_sentiment_label_v1"] == label]
            if len(rows):
                selected = str(rows.sort_values("mention_sentiment_confidence_v1", ascending=False)["context_used_v1"].iloc[0])[:500]
                if label == "positive":
                    example_positive = selected
                elif label == "negative":
                    example_negative = selected
                else:
                    example_neutral = selected

    return pd.Series({
        "mention_count": int(total),
        "review_count": int(group["review_id"].nunique(dropna=True)),
        "source_review_count": int(group["source_review_id"].nunique(dropna=True)) if "source_review_id" in group.columns else int(group["review_id"].nunique(dropna=True)),
        "positive_count": pos,
        "neutral_count": neu,
        "negative_count": neg,
        "positive_ratio": round(pos / total, 4) if total else 0.0,
        "neutral_ratio": round(neu / total, 4) if total else 0.0,
        "negative_ratio": round(neg / total, 4) if total else 0.0,
        "weighted_positive_count": round(weighted_positive_count, 4),
        "weighted_neutral_count": round(weighted_neutral_count, 4),
        "weighted_negative_count": round(weighted_negative_count, 4),
        "total_signal_weight": round(total_weight, 4),
        "smoothed_positive_ratio": round(float(smoothed_positive_ratio), 4),
        "smoothed_negative_ratio": round(float(smoothed_negative_ratio), 4),
        "avg_sentiment_value": round(float(group["sentiment_value_v1"].mean()), 4),
        "weighted_sentiment_value": round(weighted_sentiment_value, 4),
        "avg_sentiment_score": round(float(group["mention_sentiment_score_v1"].fillna(0.0).mean()), 4),
        "weighted_sentiment_score": round(weighted_sentiment_score, 4),
        "avg_sentiment_confidence": round(float(group["mention_sentiment_confidence_v1"].fillna(0.0).mean()), 4),
        "avg_aggregation_weight": round(float(weights.mean()), 4),
        "local_context_count": int(group["is_local_context_v1"].sum()),
        "review_prior_fallback_count": int(group["is_review_prior_fallback_v1"].sum()),
        "local_context_ratio": round(float(group["is_local_context_v1"].mean()), 4),
        "fallback_ratio": round(float(group["is_review_prior_fallback_v1"].mean()), 4),
        "low_confidence_flag_count": int(group["has_low_confidence_flag_v1"].sum()),
        "service_flag_count": int(group["has_service_flag_v1"].sum()),
        "price_flag_count": int(group["has_price_flag_v1"].sum()),
        "contrast_flag_count": int(group["has_contrast_flag_v1"].sum()),
        "negation_flag_count": int(group["has_negation_flag_v1"].sum()),
        "top_sentiment_flags_v1": top_flags,
        "avg_rating": round(float(group["rating_value"].fillna(0.0).mean()), 4),
        "min_rating": round(float(group["rating_value"].fillna(0.0).min()), 4),
        "max_rating": round(float(group["rating_value"].fillna(0.0).max()), 4),
        "example_positive_context_v1": example_positive,
        "example_negative_context_v1": example_negative,
        "example_neutral_context_v1": example_neutral,
    })


place_dish_signals = (
    df.groupby(GROUP_COLUMNS, dropna=False)
      .apply(aggregate_place_dish)
      .reset_index()
)

print("Pares local-plato agregados:", place_dish_signals.shape)
display(place_dish_signals.head(10))

Pares local-plato agregados: (2212, 48)


,place_id,place_name,district_id,district_name,neighborhood_id,neighborhood_name,dish_id_v1,display_dish_name_es_v1,normalized_dish_name_es_v1,dish_family_es_v1,dish_group_es_v1,mention_count,review_count,source_review_count,positive_count,neutral_count,negative_count,positive_ratio,neutral_ratio,negative_ratio,weighted_positive_count,weighted_neutral_count,weighted_negative_count,total_signal_weight,smoothed_positive_ratio,smoothed_negative_ratio,avg_sentiment_value,weighted_sentiment_value,avg_sentiment_score,weighted_sentiment_score,avg_sentiment_confidence,avg_aggregation_weight,local_context_count,review_prior_fallback_count,local_context_ratio,fallback_ratio,low_confidence_flag_count,service_flag_count,price_flag_count,contrast_flag_count,negation_flag_count,top_sentiment_flags_v1,avg_rating,min_rating,max_rating,example_positive_context_v1,example_negative_context_v1,example_neutral_context_v1
0,0046efb9-63e6-4eb0-8d95-0f93b585dbcc,Bar Pino,1e467552-511d-4adb-bd9b-257bf108e491,Macarena,73b4ba41-1e8f-41b1-aa2a-fe7975470132,PINO FLORES,dish_es_v1_03d0d732fdd6fe23,gambas,gambas,mar_y_pescado,pescado,1,1,1,1,0,0,1.0,0.0,0.0,0.9000,0.0,0.0000,0.9000,0.4872,0.2564,1.0,1.0000,1.9000,1.9000,0.9000,0.9000,1,0,1.0000,0.0000,0,0,0,0,0,[],5.00,5.0,5.0,"Todo buenísimo, las alitas fritas adobadas su santo y seña, berenjena rellena, gambas al ajillo, carnes a la brasa, pastel vegetal.......",,
1,0046efb9-63e6-4eb0-8d95-0f93b585dbcc,Bar Pino,1e467552-511d-4adb-bd9b-257bf108e491,Macarena,73b4ba41-1e8f-41b1-aa2a-fe7975470132,PINO FLORES,dish_es_v1_2c67f67ce386e9d6,carrillada,carrillada,tapas_clasicas,carrillada,1,1,1,1,0,0,1.0,0.0,0.0,0.8600,0.0,0.0000,0.8600,0.4819,0.2591,1.0,1.0000,1.4160,1.4160,0.8600,0.8600,1,0,1.0000,0.0000,0,0,0,1,0,"[contrast_marker_nearby, intensifier_nearby]",5.00,5.0,5.0,"Es difícil destacar algo de la carta , ya que todo estaba muy bueno , pero las albóndigas , las croquetas y la carrillada estaban excelentes .",,
2,0046efb9-63e6-4eb0-8d95-0f93b585dbcc,Bar Pino,1e467552-511d-4adb-bd9b-257bf108e491,Macarena,73b4ba41-1e8f-41b1-aa2a-fe7975470132,PINO FLORES,dish_es_v1_3b8c4e72931fd3d2,croqueta,croqueta,tapas_clasicas,croqueta,1,1,1,1,0,0,1.0,0.0,0.0,0.8600,0.0,0.0000,0.8600,0.4819,0.2591,1.0,1.0000,1.6709,1.6709,0.8600,0.8600,1,0,1.0000,0.0000,0,0,0,1,0,"[contrast_marker_nearby, intensifier_nearby]",5.00,5.0,5.0,"Es difícil destacar algo de la carta , ya que todo estaba muy bueno , pero las albóndigas , las croquetas y la carrillada estaban excelentes .",,
3,0046efb9-63e6-4eb0-8d95-0f93b585dbcc,Bar Pino,1e467552-511d-4adb-bd9b-257bf108e491,Macarena,73b4ba41-1e8f-41b1-aa2a-fe7975470132,PINO FLORES,dish_es_v1_4c5e83397f7a5a82,albóndigas,albondigas,carne,carne,1,1,1,1,0,0,1.0,0.0,0.0,0.8600,0.0,0.0000,0.8600,0.4819,0.2591,1.0,1.0000,1.6709,1.6709,0.8600,0.8600,1,0,1.0000,0.0000,0,0,0,1,0,"[contrast_marker_nearby, intensifier_nearby]",5.00,5.0,5.0,"Es difícil destacar algo de la carta , ya que todo estaba muy bueno , pero las albóndigas , las croquetas y la carrillada estaban excelentes .",,
4,0046efb9-63e6-4eb0-8d95-0f93b585dbcc,Bar Pino,1e467552-511d-4adb-bd9b-257bf108e491,Macarena,73b4ba41-1e8f-41b1-aa2a-fe7975470132,PINO FLORES,dish_es_v1_c325d21226b882b6,patatas bravas,patatas bravas,fritos_y_raciones,patatas,1,1,1,1,0,0,1.0,0.0,0.0,0.9000,0.0,0.0000,0.9000,0.4872,0.2564,1.0,1.0000,1.5340,1.5340,0.9000,0.9000,1,0,1.0000,0.0000,0,0,0,0,0,[],4.00,4.0,4.0,Recomiendo las patatas bravas y el serranito.,,
5,0061925b-3bc8-41fd-8bef-78ebdc871a31,Bar Hiniesta,5bd4fbf1-457b-4dcb-91f1-089ac449a5db,Casco Antiguo,e554ef0e-d9f1-4d89-8777-f92cee028f7f,SAN JULIAN,dish_es_v1_10850a8a8e314902,ensaladilla,ensaladilla,tapas_clasicas,ensaladilla,3,1,1,0,0,3,0.0,0.0,1.0,0.0000,0.0,0.9703,0.9703,0.2519,0.4963,-1.0,-1.0000,-1.6000,-2.1996,0.5467,0.3234,1,2,0.3333,0.6667,2,0,1,0,0,"[low_sentiment_confidence, no_clear_local_sentiment, used_review_prior_fallback, weak_local_sentiment_signal, price_context_detected]",1.00,1.0,1.0,,"Adjunto

In [10]:
# ============================================================
# 06. Tiers de evidencia, calidad y scoring preliminar
# ============================================================

def assign_evidence_tier(row: pd.Series) -> str:
    mention_count = int(row["mention_count"])
    review_count = int(row["review_count"])
    avg_conf = float(row["avg_sentiment_confidence"])
    local_ratio = float(row["local_context_ratio"])

    if mention_count >= 5 and review_count >= 4 and avg_conf >= 0.58 and local_ratio >= 0.45:
        return "strong"
    if mention_count >= 3 and review_count >= 2 and avg_conf >= 0.52:
        return "solid"
    if mention_count >= 2 and review_count >= 2 and avg_conf >= 0.45:
        return "emerging"
    return "weak"


def assign_signal_quality_tier(row: pd.Series) -> str:
    evidence = row["evidence_tier_v1"]
    negative_ratio = float(row["negative_ratio"])
    fallback_ratio = float(row["fallback_ratio"])
    service_count = int(row["service_flag_count"])
    price_count = int(row["price_flag_count"])
    mention_count = int(row["mention_count"])

    if evidence == "strong" and negative_ratio <= 0.25 and fallback_ratio <= 0.60:
        return "high_quality_signal"
    if evidence in {"strong", "solid"} and negative_ratio <= 0.35:
        return "usable_signal"
    if evidence == "emerging" and negative_ratio <= 0.40:
        return "emerging_signal"
    if mention_count == 1:
        return "single_mention_signal"
    if service_count > 0 or price_count > 0:
        return "noisy_signal"
    return "weak_signal"


def compute_preliminary_signal_score(row: pd.Series) -> float:
    sentiment_component = sigmoid_like(float(row["weighted_sentiment_score"]), scale=2.6)
    positive_component = clip01(float(row["smoothed_positive_ratio"]))
    negative_penalty = 1.0 - clip01(float(row["smoothed_negative_ratio"]) * 1.35)

    evidence_component = clip01(math.log1p(float(row["review_count"])) / math.log1p(6.0))
    mention_component = clip01(math.log1p(float(row["mention_count"])) / math.log1p(8.0))
    confidence_component = clip01(float(row["avg_sentiment_confidence"]))
    local_context_component = clip01(float(row["local_context_ratio"]))

    fallback_penalty = 1.0 - 0.18 * clip01(float(row["fallback_ratio"]))
    service_penalty = 0.92 if int(row["service_flag_count"]) > 0 else 1.0
    price_penalty = 0.94 if int(row["price_flag_count"]) > 0 else 1.0

    base = (
        0.28 * sentiment_component
        + 0.22 * positive_component
        + 0.20 * evidence_component
        + 0.10 * mention_component
        + 0.12 * confidence_component
        + 0.08 * local_context_component
    )

    return round(float(max(0.0, min(100.0, 100.0 * base * negative_penalty * fallback_penalty * service_penalty * price_penalty))), 4)


place_dish_signals["evidence_tier_v1"] = place_dish_signals.apply(assign_evidence_tier, axis=1)
place_dish_signals["signal_quality_tier_v1"] = place_dish_signals.apply(assign_signal_quality_tier, axis=1)
place_dish_signals["preliminary_signal_score_v1"] = place_dish_signals.apply(compute_preliminary_signal_score, axis=1)

place_dish_signals["ranking_ready_v1"] = (
    (place_dish_signals["mention_count"] >= 2)
    & (place_dish_signals["review_count"] >= 2)
    & (place_dish_signals["negative_ratio"] <= 0.45)
    & (place_dish_signals["avg_sentiment_confidence"] >= 0.45)
    & (place_dish_signals["weighted_sentiment_score"] > -0.25)
)


def assign_candidate_tier(row: pd.Series) -> str:
    if not bool(row["ranking_ready_v1"]):
        return "not_ready"

    score = float(row["preliminary_signal_score_v1"])
    evidence = row["evidence_tier_v1"]
    pos = float(row["positive_ratio"])
    neg = float(row["negative_ratio"])

    if evidence == "strong" and score >= 72 and pos >= 0.70 and neg <= 0.20:
        return "strong_candidate"
    if evidence in {"strong", "solid"} and score >= 64 and pos >= 0.62 and neg <= 0.30:
        return "promising_candidate"
    if evidence in {"solid", "emerging"} and score >= 55:
        return "weak_candidate"
    return "not_ready"


place_dish_signals["candidate_tier_v1"] = place_dish_signals.apply(assign_candidate_tier, axis=1)

place_dish_signals = place_dish_signals.sort_values(
    ["ranking_ready_v1", "preliminary_signal_score_v1", "mention_count", "positive_ratio"],
    ascending=[False, False, False, False],
).reset_index(drop=True)

print("Evidence tier counts:")
display(place_dish_signals["evidence_tier_v1"].value_counts())

print("Candidate tier counts:")
display(place_dish_signals["candidate_tier_v1"].value_counts())

display(place_dish_signals.head(30))

Evidence tier counts:


evidence_tier_v1
weak        1909
emerging     215
solid         76
strong        12
Name: count, dtype: int64

Candidate tier counts:


candidate_tier_v1
not_ready         2205
weak_candidate       7
Name: count, dtype: int64

,place_id,place_name,district_id,district_name,neighborhood_id,neighborhood_name,dish_id_v1,display_dish_name_es_v1,normalized_dish_name_es_v1,dish_family_es_v1,dish_group_es_v1,mention_count,review_count,source_review_count,positive_count,neutral_count,negative_count,positive_ratio,neutral_ratio,negative_ratio,weighted_positive_count,weighted_neutral_count,weighted_negative_count,total_signal_weight,smoothed_positive_ratio,smoothed_negative_ratio,avg_sentiment_value,weighted_sentiment_value,avg_sentiment_score,weighted_sentiment_score,avg_sentiment_confidence,avg_aggregation_weight,local_context_count,review_prior_fallback_count,local_context_ratio,fallback_ratio,low_confidence_flag_count,service_flag_count,price_flag_count,contrast_flag_count,negation_flag_count,top_sentiment_flags_v1,avg_rating,min_rating,max_rating,example_positive_context_v1,example_negative_context_v1,example_neutral_context_v1,evidence_tier_v1,signal_quality_tier_v1,preliminary_signal_score_v1,ranking_ready_v1,candidate_tier_v1
0,52d0c5c7-da1f-4733-a8a1-6d3fcac2d18d,restaurante asiático shan,e1621557-0db9-443a-8065-2fbcb8a9515f,Bellavista - La Palmera,7040f0c8-e891-4ff2-ac79-044be0f73fff,BELLAVISTA,dish_es_v1_451dcf9913bf3b32,sushi,sushi,internacional,internacional,3,3,3,3,0,0,1.0000,0.0000,0.0,2.8500,0.0000,0.0,2.8500,0.6581,0.1709,1.0000,1.0000,4.9497,4.9497,0.9500,0.9500,3,0,1.0000,0.0000,0,0,0,0,0,[intensifier_nearby],5.0000,5.0,5.0,"Comida riquísima, el sushi flambeado y el yakisoba muy bueno.",,,solid,usable_signal,62.9487,True,weak_candidate
1,9b6cc82a-5140-4158-98e4-0b2110ec92bf,Il Ristorantino Dell´Avvocato Sevilla,5bd4fbf1-457b-4dcb-91f1-089ac449a5db,Casco Antiguo,ae182ebc-43a4-46db-9364-d006cf3a73c9,ENCARNACIÓN-REGINA,dish_es_v1_1f6ccd2be75f1cc9,pizza,pizza,internacional,internacional,9,5,5,7,2,0,0.7778,0.2222,0.0,4.8419,0.1679,0.0,5.0098,0.7293,0.1248,0.7778,0.9665,1.6851,2.4826,0.6578,0.5566,5,4,0.5556,0.4444,4,0,0,1,0,"[low_sentiment_confidence, no_clear_local_sentiment, used_review_prior_fallback, weak_local_sentiment_signal, intensifier_nearby, neutral_or_ambiguous_terms_detected, contrast_...",4.2222,3.0,5.0,"Como platos principales pedimos pizzas, y fue todo un acierto: una fuera de carta con guiso de carne, increíblemente sabrosa y diferente, y otra de carta, ahumada, con un sabor...",,"Llegamos con ganas de comer una pizza y una bola de arroz rellena de carne y salsa, para nuestra sorpresa nos sentamos en mesa y nos dicen que solo hacen pizza, igual habrá que...",strong,high_quality_signal,62.1079,True,not_ready
2,a3db8f79-3216-4c52-8307-7730e9335737,Tarannà,5bd4fbf1-457b-4dcb-91f1-089ac449a5db,Casco Antiguo,903feb9b-aed1-4717-abbf-e693ac5585a0,SANTA CATALINA,dish_es_v1_ad3755cf6fd0fb67,atún,atun,mar_y_pescado,pescado,3,3,3,3,0,0,1.0000,0.0000,0.0,2.6075,0.0000,0.0,2.6075,0.6433,0.1783,1.0000,1.0000,3.1247,3.3604,0.8692,0.8692,3,0,1.0000,0.0000,0,0,0,0,0,[intensifier_nearby],4.6667,4.0,5.0,Tomamos un tartar de atún de almadraba súper recomendable y un arroz súper jugoso.,,,solid,usable_signal,60.1190,True,weak_candidate
3,ea7d6564-2b10-4bef-97e9-f91cfb33d342,Pizzería San Pablo,898f9d9e-9365-4138-a30e-d8183ddfa969,San Pablo - Santa Justa,f36f071a-eedf-4022-adcb-a0d67a0ccf28,SAN PABLO D Y E,dish_es_v1_1f6ccd2be75f1cc9,pizza,pizza,internacional,internacional,6,5,5,6,0,0,1.0000,0.0000,0.0,4.4098,0.0000,0.0,4.4098,0.7301,0.1350,1.0000,1.0000,4.6146,5.3569,0.8350,0.7350,5,1,0.8333,0.1667,1,1,1,0,0,"[intensifier_nearby, low_sentiment_confidence, no_clear_local_sentiment, used_review_prior_fallback, weak_local_sentiment_signal, price_context_detected, sentiment_may_refer_to...",5.0000,5.0,5.0,"Unas experiencia muy buena la pizza 10 de 10 me encanto, esta vez fue la carnivora y seguiré probando las demás pizza q tienen, 100 % recomendados",,,strong,high_quality_signal,60.0726,True,not_ready
4,1844aa47-4f98-47f5-a31d-0f5565cd4990,ALCÁZAR ANDALUSÍ TAPAS,5bd4fbf1-457b-4dcb-91f1-089ac449a5db,Casco Antiguo,dd0845a6-120f-4945-9a3d-ec166c36337c,FERI

In [11]:
# ============================================================
# 07. Señales globales por plato
# ============================================================

def aggregate_global_dish(group: pd.DataFrame) -> pd.Series:
    total = len(group)
    pos = int((group["mention_sentiment_label_v1"] == "positive").sum())
    neg = int((group["mention_sentiment_label_v1"] == "negative").sum())
    neu = int((group["mention_sentiment_label_v1"] == "neutral").sum())

    weights = group["aggregation_weight_v1"].fillna(0.05).clip(lower=0.03)
    weighted_score = float(np.average(group["mention_sentiment_score_v1"].fillna(0.0), weights=weights)) if total else 0.0

    return pd.Series({
        "global_mention_count": int(total),
        "global_review_count": int(group["review_id"].nunique(dropna=True)),
        "global_place_count": int(group["place_id"].nunique(dropna=True)),
        "global_neighborhood_count": int(group["neighborhood_id"].nunique(dropna=True)),
        "global_district_count": int(group["district_id"].nunique(dropna=True)),
        "global_positive_count": pos,
        "global_neutral_count": neu,
        "global_negative_count": neg,
        "global_positive_ratio": round(pos / total, 4) if total else 0.0,
        "global_negative_ratio": round(neg / total, 4) if total else 0.0,
        "global_weighted_sentiment_score": round(weighted_score, 4),
        "global_avg_confidence": round(float(group["mention_sentiment_confidence_v1"].fillna(0.0).mean()), 4),
        "global_avg_rating": round(float(group["rating_value"].fillna(0.0).mean()), 4),
    })


global_dish_signals = (
    df.groupby([
        "dish_id_v1", "display_dish_name_es_v1", "normalized_dish_name_es_v1",
        "dish_family_es_v1", "dish_group_es_v1",
    ], dropna=False)
    .apply(aggregate_global_dish)
    .reset_index()
)

max_mentions = max(float(global_dish_signals["global_mention_count"].max()), 1.0)
global_dish_signals["global_popularity_score_v1"] = global_dish_signals["global_mention_count"].apply(
    lambda x: round(math.log1p(float(x)) / math.log1p(max_mentions), 4)
)

global_dish_signals["global_dish_signal_score_v1"] = global_dish_signals.apply(
    lambda row: round(
        100.0 * (
            0.35 * sigmoid_like(float(row["global_weighted_sentiment_score"]), scale=2.6)
            + 0.25 * clip01(float(row["global_positive_ratio"]))
            + 0.20 * clip01(math.log1p(float(row["global_place_count"])) / math.log1p(20.0))
            + 0.10 * clip01(float(row["global_avg_confidence"]))
            + 0.10 * clip01(math.log1p(float(row["global_neighborhood_count"])) / math.log1p(20.0))
        ),
        4,
    ),
    axis=1,
)

global_dish_signals = global_dish_signals.sort_values(
    ["global_mention_count", "global_dish_signal_score_v1"], ascending=[False, False]
).reset_index(drop=True)

display(global_dish_signals.head(30))

,dish_id_v1,display_dish_name_es_v1,normalized_dish_name_es_v1,dish_family_es_v1,dish_group_es_v1,global_mention_count,global_review_count,global_place_count,global_neighborhood_count,global_district_count,global_positive_count,global_neutral_count,global_negative_count,global_positive_ratio,global_negative_ratio,global_weighted_sentiment_score,global_avg_confidence,global_avg_rating,global_popularity_score_v1,global_dish_signal_score_v1
0,dish_es_v1_10850a8a8e314902,ensaladilla,ensaladilla,tapas_clasicas,ensaladilla,216.0,197.0,153.0,57.0,11.0,167.0,18.0,31.0,0.7731,0.1435,1.8679,0.6275,4.1806,1.0000,83.8813
1,dish_es_v1_3b8c4e72931fd3d2,croqueta,croqueta,tapas_clasicas,croqueta,211.0,198.0,160.0,61.0,11.0,166.0,17.0,28.0,0.7867,0.1327,1.6126,0.6072,4.3128,0.9957,82.8871
2,dish_es_v1_2c67f67ce386e9d6,carrillada,carrillada,tapas_clasicas,carrillada,172.0,152.0,123.0,62.0,11.0,149.0,12.0,11.0,0.8663,0.0640,2.2245,0.6390,4.4884,0.9579,87.6920
3,dish_es_v1_0a420a9251bb5aed,solomillo al whisky,solomillo al whisky,tapas_clasicas,solomillo,167.0,152.0,126.0,57.0,11.0,132.0,10.0,25.0,0.7904,0.1497,1.9356,0.6495,4.2096,0.9524,84.8121
4,dish_es_v1_ad3755cf6fd0fb67,atún,atun,mar_y_pescado,pescado,117.0,99.0,70.0,38.0,10.0,94.0,8.0,15.0,0.8034,0.1282,1.8923,0.7066,4.4017,0.8868,85.5311
5,dish_es_v1_18ef91885a7312e4,tarta de queso,tarta de queso,postres_y_desayunos,tarta,114.0,100.0,81.0,38.0,11.0,96.0,9.0,9.0,0.8421,0.0789,2.0231,0.6626,4.4737,0.8820,86.5819
6,dish_es_v1_a646e68b8a85d929,montadito,montadito,tapas_clasicas,montadito,108.0,95.0,73.0,45.0,11.0,87.0,6.0,15.0,0.8056,0.1389,1.6205,0.5647,4.3981,0.8720,82.9715
7,dish_es_v1_c325d21226b882b6,patatas bravas,patatas bravas,fritos_y_raciones,patatas,105.0,98.0,76.0,42.0,11.0,83.0,7.0,15.0,0.7905,0.1429,1.4444,0.5900,4.1619,0.8668,81.9940
8,dish_es_v1_03d0d732fdd6fe23,gambas,gambas,mar_y_pescado,pescado,104.0,97.0,79.0,45.0,11.0,80.0,12.0,12.0,0.7692,0.1154,1.9001,0.6060,4.1635,0.8651,83.7023
9,dish_es_v1_9a1e795ae0c1d04f,hamburguesa,hamburguesa,carne,carne,90.0,75.0,51.0,36.0,11.0,63.0,8.0,19.0,0.7000,0.2111,1.3653,0.5821,3.9667,0.8385,79.2497


In [12]:
# ============================================================
# 08. Resúmenes territoriales y señales principales
# ============================================================

ranking_ready_df = place_dish_signals[place_dish_signals["ranking_ready_v1"]].copy()

summary_by_district = (
    ranking_ready_df.groupby(["district_id", "district_name"], dropna=False)
    .agg(
        place_dish_pair_count=("dish_id_v1", "count"),
        place_count=("place_id", "nunique"),
        dish_count=("dish_id_v1", "nunique"),
        total_mentions=("mention_count", "sum"),
        avg_preliminary_signal_score=("preliminary_signal_score_v1", "mean"),
        avg_positive_ratio=("positive_ratio", "mean"),
        avg_negative_ratio=("negative_ratio", "mean"),
        strong_candidate_count=("candidate_tier_v1", lambda x: int((x == "strong_candidate").sum())),
        promising_candidate_count=("candidate_tier_v1", lambda x: int((x == "promising_candidate").sum())),
        weak_candidate_count=("candidate_tier_v1", lambda x: int((x == "weak_candidate").sum())),
    )
    .reset_index()
    .sort_values(["place_dish_pair_count", "avg_preliminary_signal_score"], ascending=[False, False])
)

summary_by_neighborhood = (
    ranking_ready_df.groupby(["district_name", "neighborhood_id", "neighborhood_name"], dropna=False)
    .agg(
        place_dish_pair_count=("dish_id_v1", "count"),
        place_count=("place_id", "nunique"),
        dish_count=("dish_id_v1", "nunique"),
        total_mentions=("mention_count", "sum"),
        avg_preliminary_signal_score=("preliminary_signal_score_v1", "mean"),
        avg_positive_ratio=("positive_ratio", "mean"),
        avg_negative_ratio=("negative_ratio", "mean"),
        strong_candidate_count=("candidate_tier_v1", lambda x: int((x == "strong_candidate").sum())),
        promising_candidate_count=("candidate_tier_v1", lambda x: int((x == "promising_candidate").sum())),
        weak_candidate_count=("candidate_tier_v1", lambda x: int((x == "weak_candidate").sum())),
    )
    .reset_index()
    .sort_values(["place_dish_pair_count", "avg_preliminary_signal_score"], ascending=[False, False])
)

candidate_tier_summary = (
    place_dish_signals.groupby(["candidate_tier_v1", "evidence_tier_v1", "signal_quality_tier_v1"], dropna=False)
    .agg(
        pair_count=("dish_id_v1", "count"),
        mention_count=("mention_count", "sum"),
        review_count=("review_count", "sum"),
        avg_score=("preliminary_signal_score_v1", "mean"),
        avg_positive_ratio=("positive_ratio", "mean"),
        avg_negative_ratio=("negative_ratio", "mean"),
    )
    .reset_index()
    .sort_values(["candidate_tier_v1", "pair_count"], ascending=[True, False])
)

top_place_dish_signals = ranking_ready_df.sort_values(
    ["preliminary_signal_score_v1", "mention_count", "positive_ratio"], ascending=[False, False, False]
).head(200)

print("Ranking-ready pairs:", len(ranking_ready_df))
display(top_place_dish_signals.head(30))

print("District summary:")
display(summary_by_district.head(20))

Ranking-ready pairs: 256


,place_id,place_name,district_id,district_name,neighborhood_id,neighborhood_name,dish_id_v1,display_dish_name_es_v1,normalized_dish_name_es_v1,dish_family_es_v1,dish_group_es_v1,mention_count,review_count,source_review_count,positive_count,neutral_count,negative_count,positive_ratio,neutral_ratio,negative_ratio,weighted_positive_count,weighted_neutral_count,weighted_negative_count,total_signal_weight,smoothed_positive_ratio,smoothed_negative_ratio,avg_sentiment_value,weighted_sentiment_value,avg_sentiment_score,weighted_sentiment_score,avg_sentiment_confidence,avg_aggregation_weight,local_context_count,review_prior_fallback_count,local_context_ratio,fallback_ratio,low_confidence_flag_count,service_flag_count,price_flag_count,contrast_flag_count,negation_flag_count,top_sentiment_flags_v1,avg_rating,min_rating,max_rating,example_positive_context_v1,example_negative_context_v1,example_neutral_context_v1,evidence_tier_v1,signal_quality_tier_v1,preliminary_signal_score_v1,ranking_ready_v1,candidate_tier_v1
0,52d0c5c7-da1f-4733-a8a1-6d3fcac2d18d,restaurante asiático shan,e1621557-0db9-443a-8065-2fbcb8a9515f,Bellavista - La Palmera,7040f0c8-e891-4ff2-ac79-044be0f73fff,BELLAVISTA,dish_es_v1_451dcf9913bf3b32,sushi,sushi,internacional,internacional,3,3,3,3,0,0,1.0000,0.0000,0.0,2.8500,0.0000,0.0,2.8500,0.6581,0.1709,1.0000,1.0000,4.9497,4.9497,0.9500,0.9500,3,0,1.0000,0.0000,0,0,0,0,0,[intensifier_nearby],5.0000,5.0,5.0,"Comida riquísima, el sushi flambeado y el yakisoba muy bueno.",,,solid,usable_signal,62.9487,True,weak_candidate
1,9b6cc82a-5140-4158-98e4-0b2110ec92bf,Il Ristorantino Dell´Avvocato Sevilla,5bd4fbf1-457b-4dcb-91f1-089ac449a5db,Casco Antiguo,ae182ebc-43a4-46db-9364-d006cf3a73c9,ENCARNACIÓN-REGINA,dish_es_v1_1f6ccd2be75f1cc9,pizza,pizza,internacional,internacional,9,5,5,7,2,0,0.7778,0.2222,0.0,4.8419,0.1679,0.0,5.0098,0.7293,0.1248,0.7778,0.9665,1.6851,2.4826,0.6578,0.5566,5,4,0.5556,0.4444,4,0,0,1,0,"[low_sentiment_confidence, no_clear_local_sentiment, used_review_prior_fallback, weak_local_sentiment_signal, intensifier_nearby, neutral_or_ambiguous_terms_detected, contrast_...",4.2222,3.0,5.0,"Como platos principales pedimos pizzas, y fue todo un acierto: una fuera de carta con guiso de carne, increíblemente sabrosa y diferente, y otra de carta, ahumada, con un sabor...",,"Llegamos con ganas de comer una pizza y una bola de arroz rellena de carne y salsa, para nuestra sorpresa nos sentamos en mesa y nos dicen que solo hacen pizza, igual habrá que...",strong,high_quality_signal,62.1079,True,not_ready
2,a3db8f79-3216-4c52-8307-7730e9335737,Tarannà,5bd4fbf1-457b-4dcb-91f1-089ac449a5db,Casco Antiguo,903feb9b-aed1-4717-abbf-e693ac5585a0,SANTA CATALINA,dish_es_v1_ad3755cf6fd0fb67,atún,atun,mar_y_pescado,pescado,3,3,3,3,0,0,1.0000,0.0000,0.0,2.6075,0.0000,0.0,2.6075,0.6433,0.1783,1.0000,1.0000,3.1247,3.3604,0.8692,0.8692,3,0,1.0000,0.0000,0,0,0,0,0,[intensifier_nearby],4.6667,4.0,5.0,Tomamos un tartar de atún de almadraba súper recomendable y un arroz súper jugoso.,,,solid,usable_signal,60.1190,True,weak_candidate
3,ea7d6564-2b10-4bef-97e9-f91cfb33d342,Pizzería San Pablo,898f9d9e-9365-4138-a30e-d8183ddfa969,San Pablo - Santa Justa,f36f071a-eedf-4022-adcb-a0d67a0ccf28,SAN PABLO D Y E,dish_es_v1_1f6ccd2be75f1cc9,pizza,pizza,internacional,internacional,6,5,5,6,0,0,1.0000,0.0000,0.0,4.4098,0.0000,0.0,4.4098,0.7301,0.1350,1.0000,1.0000,4.6146,5.3569,0.8350,0.7350,5,1,0.8333,0.1667,1,1,1,0,0,"[intensifier_nearby, low_sentiment_confidence, no_clear_local_sentiment, used_review_prior_fallback, weak_local_sentiment_signal, price_context_detected, sentiment_may_refer_to...",5.0000,5.0,5.0,"Unas experiencia muy buena la pizza 10 de 10 me encanto, esta vez fue la carnivora y seguiré probando las demás pizza q tienen, 100 % recomendados",,,strong,high_quality_signal,60.0726,True,not_ready
4,1844aa47-4f98-47f5-a31d-0f5565cd4990,ALCÁZAR ANDALUSÍ TAPAS,5bd4fbf1-457b-4dcb-91f1-089ac449a5db,Casco Antiguo,dd0845a6-120f-4945-9a3d-ec166c36337c,FERI

District summary:


,district_id,district_name,place_dish_pair_count,place_count,dish_count,total_mentions,avg_preliminary_signal_score,avg_positive_ratio,avg_negative_ratio,strong_candidate_count,promising_candidate_count,weak_candidate_count
3,5bd4fbf1-457b-4dcb-91f1-089ac449a5db,Casco Antiguo,76,51,25,204,43.261388,0.887051,0.005263,0,0,3
2,22b433e4-3b14-4b09-bfaa-f0ca1d7e6bdc,Sur,24,16,18,62,43.208275,0.906946,0.000000,0,0,1
5,7f42c0b3-14d1-400c-bcde-c128faafc4c5,Este - Alcosa - Torreblanca,24,16,15,65,41.081896,0.883929,0.005954,0,0,0
4,6af93f2e-b5da-4456-9cb1-e68958a97dbf,Nervión,22,12,15,57,41.211332,0.965909,0.011364,0,0,0
9,f9420edb-f021-42df-a2cc-e4bb21d0eceb,Triana,20,14,12,47,41.657275,0.933330,0.000000,0,0,0
1,1e467552-511d-4adb-bd9b-257bf108e491,Macarena,18,15,13,56,41.202761,0.951389,0.000000,0,0,1
0,09b2875d-a826-4c21-bd78-1d8fa48009de,Cerro - Amate,17,13,9,49,45.918788,0.945100,0.031371,0,0,1
7,af2fbf25-5491-4c27-811f-28b7a400ea84,Norte,17,12,15,52,41.378376,0.973859,0.019606,0,0,0
6,898f9d9e-9365-4138-a30e-d8183ddfa969,San Pablo - Santa Justa,16,15,14,40,43.516531,0.916669,0.020831,0,0,0
8,e1621557-0db9-443a-8065-2fbcb8a9515f,Bellavista - La Palmera,16,16,11,37,40.526519,0.906250,0.000000,0,0,1


In [13]:
# ============================================================
# 09. Señales para revisión manual
# ============================================================

manual_review_low_evidence = (
    place_dish_signals[
        (place_dish_signals["preliminary_signal_score_v1"] >= 60)
        & (place_dish_signals["evidence_tier_v1"].isin(["weak", "emerging"]))
    ]
    .sort_values(["preliminary_signal_score_v1", "mention_count"], ascending=[False, False])
    .head(150)
)

manual_review_conflict = (
    place_dish_signals[
        (place_dish_signals["mention_count"] >= 2)
        & (
            (place_dish_signals["negative_ratio"] >= 0.35)
            | (place_dish_signals["fallback_ratio"] >= 0.75)
            | (place_dish_signals["service_flag_count"] > 0)
        )
    ]
    .sort_values(["mention_count", "negative_ratio", "fallback_ratio"], ascending=[False, False, False])
    .head(150)
)

manual_review_signals = pd.concat([
    manual_review_low_evidence.assign(manual_review_reason_v1="high_score_low_evidence"),
    manual_review_conflict.assign(manual_review_reason_v1="conflict_or_noisy_signal"),
], ignore_index=True).drop_duplicates(subset=["place_id", "dish_id_v1"], keep="first")

print("Señales para revisión manual:", len(manual_review_signals))
display(manual_review_signals.head(30))

Señales para revisión manual: 150


,place_id,place_name,district_id,district_name,neighborhood_id,neighborhood_name,dish_id_v1,display_dish_name_es_v1,normalized_dish_name_es_v1,dish_family_es_v1,dish_group_es_v1,mention_count,review_count,source_review_count,positive_count,neutral_count,negative_count,positive_ratio,neutral_ratio,negative_ratio,weighted_positive_count,weighted_neutral_count,weighted_negative_count,total_signal_weight,smoothed_positive_ratio,smoothed_negative_ratio,avg_sentiment_value,weighted_sentiment_value,avg_sentiment_score,weighted_sentiment_score,avg_sentiment_confidence,avg_aggregation_weight,local_context_count,review_prior_fallback_count,local_context_ratio,fallback_ratio,low_confidence_flag_count,service_flag_count,price_flag_count,contrast_flag_count,negation_flag_count,top_sentiment_flags_v1,avg_rating,min_rating,max_rating,example_positive_context_v1,example_negative_context_v1,example_neutral_context_v1,evidence_tier_v1,signal_quality_tier_v1,preliminary_signal_score_v1,ranking_ready_v1,candidate_tier_v1,manual_review_reason_v1
0,4d74b42b-54ef-4d31-9379-7df53dd7f6f6,Bendito Placer,09b2875d-a826-4c21-bd78-1d8fa48009de,Cerro - Amate,7feeccb6-1f5d-47c2-a067-02dc6466c762,EL CERRO,dish_es_v1_9a1e795ae0c1d04f,hamburguesa,hamburguesa,carne,carne,10,5,5,5,1,4,0.5000,0.1000,0.4000,1.2418,0.0840,3.4255,4.7512,0.2892,0.5709,0.1000,-0.4596,-0.3905,-1.3944,0.5956,0.4751,5,5,0.5000,0.5000,5,0,0,2,1,"[low_sentiment_confidence, no_clear_local_sentiment, used_review_prior_fallback, weak_local_sentiment_signal, contrast_marker_nearby, intensifier_nearby, mixed_local_evidence, ...",3.2000,1.0,5.0,"n mucho sabor y se distinguían muy bien los sabores, y la especia que llevaba le daba muchiiisimo sabor, no ha gustado mucho; y de hamburguesas pedimos la Flamenca que estaba b...","La otra hamburguesa, la “Bendición Suprema”, resultó aún más decepcionante: apenas llevaba queso, el bacon seguía crudo y el chocolate blanco con topping de almendras y coco si...","Pedimos estas dos hamburguesas, la bendito placer y la templaria.",strong,weak_signal,11.0715,False,not_ready,conflict_or_noisy_signal
1,775735df-c2ed-4908-93c6-cc31da992abb,Heladería Valentina,6af93f2e-b5da-4456-9cb1-e68958a97dbf,Nervión,b63a69e4-c0a5-4a2b-b936-97511845b8d8,HUERTA DEL PILAR,dish_es_v1_d35dbc1062811528,helado,helado,postres_y_desayunos,postre,9,4,4,5,0,4,0.5556,0.0000,0.4444,1.1583,0.0000,3.3865,4.5448,0.2861,0.5814,0.1111,-0.4903,0.0168,-0.7556,0.6170,0.5050,5,4,0.5556,0.4444,4,0,0,0,0,"[intensifier_nearby, low_sentiment_confidence, no_clear_local_sentiment, used_review_prior_fallback, weak_local_sentiment_signal]",3.2222,1.0,5.0,La calidad del helado y el cucurucho son buenas.,"Lamentablemente, la cantidad es ridícula porque el helado está tan duro que no pueden presentarlo bien sobre el cucurucho.",,strong,weak_signal,10.8306,False,not_ready,conflict_or_noisy_signal
2,6ee0277a-c373-4d0d-94bc-f7d90766cdc2,I Love Churros - Cafetería churreria en Ronda de Capuchinos,1e467552-511d-4adb-bd9b-257bf108e491,Macarena,ec7b7bef-c772-438a-9318-cc5ee459a6b9,CRUZ ROJA-CAPUCHINOS,dish_es_v1_da687fa2af0b4205,churros,churros,postres_y_desayunos,postre,8,5,5,7,1,0,0.8750,0.1250,0.0000,3.5914,0.4800,0.0000,4.0714,0.6493,0.1414,0.8750,0.8821,1.3752,1.6476,0.6325,0.5089,5,3,0.6250,0.3750,3,1,0,3,0,"[contrast_marker_nearby, low_sentiment_confidence, no_clear_local_sentiment, used_review_prior_fallback, weak_local_sentiment_signal, intensifier_nearby, sentiment_may_refer_to...",4.7500,3.0,5.0,Increíble lugar para comer churros y chocolate!,,"El chocolate y los churros normal, nada extraordinario pero están buenos.",strong,high_quality_signal,53.5446,True,not_ready,conflict_or_noisy_signal
3,3b20f382-a6d0-4c7d-8e70-6d0ce11eee18,UDON Viapol,6af93f2e-b5da-4456-9cb1-e68958a97dbf,Nervión,00162efb-57bb-451d-a462-2bad40efc8b0,LA BUHAIRA,dish_es_v1_830869f3d7f1fb36,ramen,ramen,otros,ramen,7,1,1,1,1,5,0.1429,0.1429,0.7143,0.6749,0.4800,2.1438,3.2987,0.2659,0.4991,-0.5714,-0.4453,-1.2871,-1.7141,0.5793,0.4712,4

In [14]:
# ============================================================
# 10. Checks finales
# ============================================================

checks = {
    "has_place_dish_signals": len(place_dish_signals) > 0,
    "has_ranking_ready_signals": bool(place_dish_signals["ranking_ready_v1"].any()),
    "all_signals_have_place": bool(place_dish_signals["place_id"].notna().all()),
    "all_signals_have_dish": bool(place_dish_signals["dish_id_v1"].notna().all()),
    "all_signals_have_neighborhood": bool(place_dish_signals["neighborhood_id"].notna().all()),
    "all_signals_have_district": bool(place_dish_signals["district_id"].notna().all()),
    "no_negative_mention_counts": bool((place_dish_signals["mention_count"] >= 0).all()),
    "no_negative_review_counts": bool((place_dish_signals["review_count"] >= 0).all()),
    "score_in_0_100": bool(place_dish_signals["preliminary_signal_score_v1"].between(0, 100).all()),
    "has_global_dish_signals": len(global_dish_signals) > 0,
    "global_dish_ids_are_unique": bool(global_dish_signals["dish_id_v1"].nunique(dropna=True) == len(global_dish_signals)),
}

print(json.dumps(checks, indent=2, ensure_ascii=False))
failed = [k for k, v in checks.items() if not bool(v)]
if failed:
    print("Checks fallidos:", failed)
else:
    print("Todos los checks principales han pasado.")

{
  "has_place_dish_signals": true,
  "has_ranking_ready_signals": true,
  "all_signals_have_place": true,
  "all_signals_have_dish": true,
  "all_signals_have_neighborhood": true,
  "all_signals_have_district": true,
  "no_negative_mention_counts": true,
  "no_negative_review_counts": true,
  "score_in_0_100": true,
  "has_global_dish_signals": true,
  "global_dish_ids_are_unique": true
}
Todos los checks principales han pasado.


In [15]:
# ============================================================
# 11. Guardado de artefactos
# ============================================================

signals_csv_path = OUTPUT_DIR / "sevilla_place_dish_signals_v1.csv"
signals_jsonl_path = OUTPUT_DIR / "sevilla_place_dish_signals_v1.jsonl"

ranking_candidates_csv_path = OUTPUT_DIR / "sevilla_place_dish_ranking_candidates_v1.csv"
ranking_candidates_jsonl_path = OUTPUT_DIR / "sevilla_place_dish_ranking_candidates_v1.jsonl"

global_dish_csv_path = OUTPUT_DIR / "sevilla_global_dish_signals_v1.csv"
district_csv_path = OUTPUT_DIR / "sevilla_place_dish_signals_by_district_v1.csv"
neighborhood_csv_path = OUTPUT_DIR / "sevilla_place_dish_signals_by_neighborhood_v1.csv"
tier_summary_csv_path = OUTPUT_DIR / "sevilla_place_dish_signal_tier_summary_v1.csv"
top_signals_csv_path = OUTPUT_DIR / "sevilla_top_place_dish_signals_v1.csv"
manual_review_csv_path = OUTPUT_DIR / "sevilla_place_dish_signals_manual_review_v1.csv"
summary_json_path = OUTPUT_DIR / "sevilla_place_dish_signal_summary_v1.json"

signals_csv_df = place_dish_signals.copy()
signals_csv_df["top_sentiment_flags_v1"] = signals_csv_df["top_sentiment_flags_v1"].apply(
    lambda x: json.dumps(x, ensure_ascii=False) if isinstance(x, list) else x
)

ranking_candidates = place_dish_signals[place_dish_signals["ranking_ready_v1"]].copy()
ranking_candidates_csv_df = ranking_candidates.copy()
ranking_candidates_csv_df["top_sentiment_flags_v1"] = ranking_candidates_csv_df["top_sentiment_flags_v1"].apply(
    lambda x: json.dumps(x, ensure_ascii=False) if isinstance(x, list) else x
)

manual_review_csv_df = manual_review_signals.copy()
if len(manual_review_csv_df) and "top_sentiment_flags_v1" in manual_review_csv_df.columns:
    manual_review_csv_df["top_sentiment_flags_v1"] = manual_review_csv_df["top_sentiment_flags_v1"].apply(
        lambda x: json.dumps(x, ensure_ascii=False) if isinstance(x, list) else x
    )

signals_csv_df.to_csv(signals_csv_path, index=False, encoding="utf-8")
save_jsonl(place_dish_signals, signals_jsonl_path)

ranking_candidates_csv_df.to_csv(ranking_candidates_csv_path, index=False, encoding="utf-8")
save_jsonl(ranking_candidates, ranking_candidates_jsonl_path)

global_dish_signals.to_csv(global_dish_csv_path, index=False, encoding="utf-8")
summary_by_district.to_csv(district_csv_path, index=False, encoding="utf-8")
summary_by_neighborhood.to_csv(neighborhood_csv_path, index=False, encoding="utf-8")
candidate_tier_summary.to_csv(tier_summary_csv_path, index=False, encoding="utf-8")
top_place_dish_signals.to_csv(top_signals_csv_path, index=False, encoding="utf-8")
manual_review_csv_df.to_csv(manual_review_csv_path, index=False, encoding="utf-8")

summary = {
    "notebook": "16_sevilla_place_dish_signal_aggregation",
    "version": "sevilla_place_dish_signal_aggregation_v1",
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "input_path": str(INPUT_PATH),
    "output_dir": str(OUTPUT_DIR),
    "input": {
        "total_mentions": int(len(df)),
        "unique_reviews": int(df["review_id"].nunique(dropna=True)),
        "unique_places": int(df["place_id"].nunique(dropna=True)),
        "unique_dishes": int(df["dish_id_v1"].nunique(dropna=True)),
        "unique_neighborhoods": int(df["neighborhood_id"].nunique(dropna=True)),
        "unique_districts": int(df["district_id"].nunique(dropna=True)),
        "sentiment_counts": df["mention_sentiment_label_v1"].value_counts().to_dict(),
    },
    "aggregation": {
        "total_place_dish_pairs": int(len(place_dish_signals)),
        "ranking_ready_pairs": int(place_dish_signals["ranking_ready_v1"].sum()),
        "not_ranking_ready_pairs": int((~place_dish_signals["ranking_ready_v1"]).sum()),
        "unique_places_with_signal": int(place_dish_signals["place_id"].nunique(dropna=True)),
        "unique_dishes_with_signal": int(place_dish_signals["dish_id_v1"].nunique(dropna=True)),
        "unique_neighborhoods_with_signal": int(place_dish_signals["neighborhood_id"].nunique(dropna=True)),
        "unique_districts_with_signal": int(place_dish_signals["district_id"].nunique(dropna=True)),
        "evidence_tier_counts": place_dish_signals["evidence_tier_v1"].value_counts().to_dict(),
        "signal_quality_tier_counts": place_dish_signals["signal_quality_tier_v1"].value_counts().to_dict(),
        "candidate_tier_counts": place_dish_signals["candidate_tier_v1"].value_counts().to_dict(),
        "preliminary_signal_score": {
            "min": float(place_dish_signals["preliminary_signal_score_v1"].min()) if len(place_dish_signals) else 0.0,
            "mean": float(place_dish_signals["preliminary_signal_score_v1"].mean()) if len(place_dish_signals) else 0.0,
            "median": float(place_dish_signals["preliminary_signal_score_v1"].median()) if len(place_dish_signals) else 0.0,
            "max": float(place_dish_signals["preliminary_signal_score_v1"].max()) if len(place_dish_signals) else 0.0,
        },
        "manual_review_signal_count": int(len(manual_review_signals)),
    },
    "global_dish_signals": {
        "dish_count": int(len(global_dish_signals)),
        "top_global_dishes": global_dish_signals.head(30).replace({np.nan: None}).to_dict(orient="records"),
    },
    "top_place_dish_signals": top_place_dish_signals.head(30).replace({np.nan: None}).to_dict(orient="records"),
    "checks": checks,
    "artifacts": {
        "signals_csv": signals_csv_path.name,
        "signals_jsonl": signals_jsonl_path.name,
        "ranking_candidates_csv": ranking_candidates_csv_path.name,
        "ranking_candidates_jsonl": ranking_candidates_jsonl_path.name,
        "global_dish_signals_csv": global_dish_csv_path.name,
        "district_summary_csv": district_csv_path.name,
        "neighborhood_summary_csv": neighborhood_csv_path.name,
        "tier_summary_csv": tier_summary_csv_path.name,
        "top_signals_csv": top_signals_csv_path.name,
        "manual_review_csv": manual_review_csv_path.name,
    },
}

save_json(summary, summary_json_path)

print("Artefactos guardados en:", OUTPUT_DIR)
print("Resumen:", summary_json_path)
print(json.dumps({
    "total_place_dish_pairs": len(place_dish_signals),
    "ranking_ready_pairs": int(place_dish_signals["ranking_ready_v1"].sum()),
    "candidate_tier_counts": place_dish_signals["candidate_tier_v1"].value_counts().to_dict(),
    "evidence_tier_counts": place_dish_signals["evidence_tier_v1"].value_counts().to_dict(),
}, indent=2, ensure_ascii=False))

Artefactos guardados en: c:\Users\USUARIO\Documents\Proyectos_Master_IA_Big_Data\hidden-gems-pipeline\data\artifacts\ai\sevilla\aggregation
Resumen: c:\Users\USUARIO\Documents\Proyectos_Master_IA_Big_Data\hidden-gems-pipeline\data\artifacts\ai\sevilla\aggregation\sevilla_place_dish_signal_summary_v1.json
{
  "total_place_dish_pairs": 2212,
  "ranking_ready_pairs": 256,
  "candidate_tier_counts": {
    "not_ready": 2205,
    "weak_candidate": 7
  },
  "evidence_tier_counts": {
    "weak": 1909,
    "emerging": 215,
    "solid": 76,
    "strong": 12
  }
}
